# 1. LIBRARIES

In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path to allow imports from functions_final.py
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib as mpl
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

# Import functions from functions_final.py
from functions_final import *
from UQpy.distributions import Uniform, JointIndependent

# Set matplotlib parameters for better aesthetics
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False,
                        'figure.dpi': 100
                    })

# 2. LOADING THE CARBONATION MODEL

### 2.1 Load

In [2]:
# Path to the trained model
# name_best_model = r'D:\Documentos\ic_victor\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'
# name_best_model = r'/home/wmpjrufg/Documents/2024-1_victor_hugo_renata_maria/beam_problem_1/model_NeuralNetwork_MLP_fold_4.pkl'
# name_best_model = r'D:\py\2024-1_victor_hugo_renata_maria\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'
name_best_model = r'D:\github\2024-1_victor_hugo_renata_maria\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'

# Load the model
model = joblib.load(name_best_model)
print("Carbonation model loaded successfully!")
print(f"   Expected features: {model.feature_names_in_}")

Carbonation model loaded successfully!
   Expected features: ['CO2 (%)' 'fc (MPa)' 'RH (%)' 'Type of cement' 'Exposure conditions'
 't (years)']


### 2.2 Testing model

In [3]:
mat = {
        'f_ck [MPa]': 30,
        'Type of cement': 2
      }
expo = {
          'Installation year': 1990,
          'Exposure conditions': 2,
          'Relative humidity [%]': 40
        }
geo = {'cover[mm]':30}
load = {}
predictor = CO2Predictor()
beam_with_rh = Beam(geo=geo, mat=mat, load=load, expo=expo)
predictor.set_beam(beam_with_rh)
profile = predictor.carbonation_profile(model_=model, lifetime=150)

In [4]:
profile

,calendar year,t (years),CO2 (%),carbonation depth (mm)
0,1990,0,0.03574,0.589410
1,2000,10,0.03690,10.735841
2,2010,20,0.03893,15.324331
3,2020,30,0.04132,18.963275
4,2030,40,0.04407,21.881557
5,2040,50,0.04718,24.647107
6,2050,60,0.05065,27.325433
7,2060,70,0.05448,29.712950
8,2070,80,0.05867,31.815637
9,2080,90,0.06322,33.933596


In [5]:
carb_depth_mm = predictor.carbonation_depth_at_time(profile, 2025)
carb_depth_mm

20.42241603415687

# 3. DEFINITION OF RANDOM VARIABLES

Defines the probability distributions for the input variables:
- Concrete compressive strength ($f_{ck}$);
- Relative humidity (RH).
- Cover depth (cov).

### 3.1 Design variables

In [6]:
fck_min = 20 # MPa
fck_max = 50 # MPa
rh_min  = 50 # %
rh_max  = 80 # %
cov_min = 2  # mm
cov_max = 6  # mm

### 3.2 Fixed parameters for the analysis

In [7]:
cement_type          = 3
installation_year    = 1990
exposure_conditions  = 2
n_samples            = 50         # Number of design samples. Use 1 for testing one sample
n_latent_samples     = 500        # Number of latent samples per design sample
n_samples_validation = 3          # Number of validation samples
n_lambdas            = 4          # Number of λs to be predicted (λ1, λ2, λ3, λ4)

### 3.3 Samples

In [8]:
# Distributions of random variables
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist   = Uniform(loc=cov_min, scale=cov_max - cov_min)

# Joint distribution
joint = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

# Generate samples
x_pce_rvs = joint.rvs(n_samples)

# Report sample statistics
print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations: {n_samples * n_latent_samples}")
print("\nSample statistics:")
print(f"   fck:   {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:    {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover:   {x_pce_rvs[:, 2].min():.3f} - {x_pce_rvs[:, 2].max():.3f} mm (mean: {x_pce_rvs[:, 2].mean():.3f} mm)")

Samples generated successfully!
   Number of design samples: 50
   Number of latent samples per design sample: 500
   Total simulations: 25000

Sample statistics:
   fck:   20.1 - 49.0 MPa (mean: 33.3 MPa)
   RH:    50.2 - 79.3% (mean: 66.9%)
   cover:   2.061 - 5.993 mm (mean: 4.242 mm)


# 4. EVALUATION OF CARBONATION PROGRESS USING GENERALIZED LAMBDA DISTRIBUTION

### 4.1 Step time

In [ ]:
times               = np.arange(0, 150, 20)
times               = [10]
complete_model_list = []
pce_dataset         = []

### 4.2 Loop over design samples

In [ ]:
print("="*60)
print("BUILDING THE DURABILITY EMULATOR")
print("="*60)
dfs = []

for t in times:
    print(f'\n{"-"*40}')
    print(f'PROCESSING TIME: {t} years')
    print(f'{"-"*40}')

    # ============================================================
    # EMULATOR FUNCTION - CARBONATION DEPTH AND LAMBDAS
    # ============================================================
    df = emulator_function_time_durability(
                                                x=x_pce_rvs,
                                                names_x_variables=["fck", "rh", "cov"],
                                                carb_model=model,
                                                cement_type=cement_type,
                                                installation_year=installation_year,
                                                exposure_conditions=exposure_conditions,
                                                time_step=t,
                                                n_latent_samples=n_latent_samples,
                                                verbose=False
                                            )
    dfs.append(df)
    # ============================================================
    # SAVE AND DISPLAY THE RESULTS FOR THIS TIME STEP
    # ============================================================

    print(f"\n   📊 DEMONSTRAÇÃO DOS DADOS PARA t = {t} anos:")
    print(f"   {'='*50}")
    samples_summary = df.groupby(['fck', 'rh', 'cov'], sort=False).agg({
                                                                            'Carbonation_depth_mm': ['mean', 'std'],
                                                                            'lambda 1': 'mean',
                                                                            'lambda 2': 'mean', 
                                                                            'lambda 3': 'mean',
                                                                            'lambda 4': 'mean'
                                                                        }).round(4)
                                    
#     complete_model_list.append(df)
#     samples_summary.columns = ['Carbonation_mean_mm', 'Carbonation_std_mm', 'λ1_mean', 'λ2_mean', 'λ3_mean', 'λ4_mean']
#     samples_summary = samples_summary.reset_index()
#     y_pce_rvs = samples_summary[['λ1_mean', 'λ2_mean', 'λ3_mean', 'λ4_mean']].values
#     pce_dataset.append(samples_summary)

#     # Mostrar cada sample na ordem original
#     for idx, row in samples_summary.iterrows():
#         print(f"\n   📌 SAMPLE {idx+1} (ordem original da função):")
#         print(f"      Entradas (X):")
#         print(f"        - fck:   {row['fck']:.2f} kPa")
#         print(f"        - RH:    {row['rh']:.2f} %")
#         print(f"      Carbonatação:")
#         print(f"        - Média: {row['Carbonation_mean_mm']:.4f} mm")
#         print(f"        - Desvio: {row['Carbonation_std_mm']:.4f} mm")
#         print(f"      Saídas (Y) - Lambdas:")
#         print(f"        - λ1: {row['λ1_mean']:.6f}")
#         print(f"        - λ2: {row['λ2_mean']:.6f}")
#         print(f"        - λ3: {row['λ3_mean']:.6f}")
#         print(f"        - λ4: {row['λ4_mean']:.6f}")


#     # =============================================================
#     # BUILDING THE PCE METAMODEL
#     # =============================================================
#     max_degree       = 3
#     polynomial_basis = TotalDegreeBasis(joint, max_degree)
#     least_squares    = LeastSquareRegression()
#     pce_metamodel    = PolynomialChaosExpansion(polynomial_basis=polynomial_basis, regression_method=least_squares)
    
#     # Remove linhas com NaN em x ou y
#     mask = ~np.isnan(y_pce_rvs).any(axis=1)
#     x_pce_rvs = x_pce_rvs[mask]
#     y_pce_rvs = y_pce_rvs[mask]
                                                                                                            
#     # Train
#     pce_metamodel.fit(x_pce_rvs, y_pce_rvs)
#     print("x_pce_rvs", x_pce_rvs)
#     print("y_pce_rvs", y_pce_rvs)

#     # Save the PCE metamodel for this time step
#     filename = f'pce_metamodel_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
#     with open(filename, 'wb') as f:
#         dill.dump(pce_metamodel, f)
#     print(f"   💾 PCE metamodel saved as '{filename}'")

#     # Validation process
#     x_pce_rvs_val = joint.rvs(n_samples_validation)
#     df_val        = emulator_function_time_durability(
#                                                         x=x_pce_rvs_val,
#                                                         names_x_variables=["fck", "rh"],
#                                                         carb_model=model,
#                                                         cement_type=cement_type,
#                                                         installation_year=installation_year,
#                                                         exposure_conditions=exposure_conditions,
#                                                         time_step=t,
#                                                         n_latent_samples=n_latent_samples,
#                                                         verbose=True
#                                                     )
#     samples_summary_val = df_val.groupby(['fck', 'rh'], sort=False).agg({
#                                                                     'Carbonation_depth_mm': ['mean', 'std'],
#                                                                     'lambda 1': 'mean',
#                                                                     'lambda 2': 'mean', 
#                                                                     'lambda 3': 'mean',
#                                                                     'lambda 4': 'mean'
#                                                                 }).round(4)
#     samples_summary_val.columns = ['Carbonation_mean_mm', 'Carbonation_std_mm', 'λ1_mean', 'λ2_mean', 'λ3_mean', 'λ4_mean']
#     samples_summary_val = samples_summary_val.reset_index()
#     y_pce_val = samples_summary_val[['λ1_mean', 'λ2_mean', 'λ3_mean', 'λ4_mean']].values
#     print("y_pce_val", y_pce_val)
#     mask = ~np.isnan(y_pce_val).any(axis=1)
#     x_pce_rvs_val = x_pce_rvs_val[mask]
#     y_pce_val = y_pce_val[mask]
#     y_pce_pre = pce_metamodel.predict(x_pce_rvs_val)
#     print("y_pce_pre", y_pce_pre)
#     mse_por_lambda = []
#     r2_por_lambda  = []
#     for ii in range(n_lambdas):
#         verdade = y_pce_val[:, ii]
#         predito = y_pce_pre[:, ii]
        
#         # Calcular MSE
#         print("verdade", verdade)
#         print("predito", predito)
#         mse = mean_squared_error(verdade, predito)
#         mse_por_lambda.append(mse)
        
#         # Calcular R²
#         r2 = r2_score(verdade, predito)
#         r2_por_lambda.append(r2)
#     statistics_ = pd.DataFrame({'install year': installation_year, 't0+ (year)': t, 'MSE λ1': mse_por_lambda[0], 'MSE λ2': mse_por_lambda[1], 'MSE λ3': mse_por_lambda[2], 'MSE λ4': mse_por_lambda[3], 'R² λ1': r2_por_lambda[0], 'R² λ2': r2_por_lambda[1], 'R² λ3': r2_por_lambda[2], 'R² λ4': r2_por_lambda[3]}, index=[0])
#     filename_stats = f'pce_validation_stats_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.xlsx'
#     with open(filename_stats, 'wb') as f:
#         dill.dump(statistics_, f)
#     print(f"   💾 PCE validation statistics saved as '{filename_stats}'")


# # # Save concatenated DataFrame
# results_df = pd.concat(complete_model_list, ignore_index=True)
# results_df.to_excel('complete_model_durability_analysis_results.xlsx', index=False)
# pce_dataset_df = pd.concat(pce_dataset, ignore_index=True)
# pce_dataset_df.to_excel('pce_dataset_durability_analysis_results.xlsx', index=False)
# # results_df.to_pickle('durability_analysis_results.pkl')

# # print("\n📁 Results saved to:")
# # print("   - durability_analysis_results.xlsx")
# # print("   - durability_analysis_results.pkl")

BUILDING THE DURABILITY EMULATOR

----------------------------------------
PROCESSING TIME: 10 years
----------------------------------------


In [14]:
dfs[0]

,fck,rh,cov,RH_latent,RH_effective,FCK_latent,FCK_effective,cov_latent,cov_effective,Carbonation_depth_mm,Time (years),g,lambda 1,lambda 2,lambda 3,lambda 4
0,39.966600,79.318727,4.445598,1.030706,81.754290,0.985596,39.390936,0.991858,4.409400,4.037419,10,0.371981,0.193284,1.489422,0.095564,0.33572
1,39.966600,79.318727,4.445598,1.011576,80.236935,1.036986,41.444817,0.999758,4.444522,3.901814,10,0.542707,0.193284,1.489422,0.095564,0.33572
2,39.966600,79.318727,4.445598,1.012244,80.289881,1.005290,40.178028,1.040590,4.626042,4.100907,10,0.525135,0.193284,1.489422,0.095564,0.33572
3,39.966600,79.318727,4.445598,0.926820,73.514210,1.091157,43.609834,1.051860,4.676146,4.528856,10,0.147291,0.193284,1.489422,0.095564,0.33572
4,39.966600,79.318727,4.445598,1.020948,80.980277,1.159799,46.353224,1.040932,4.627566,3.198344,10,1.429222,0.193284,1.489422,0.095564,0.33572
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,41.531201,68.534175,2.921576,1.060889,72.707130,1.036761,43.057947,1.056823,3.087589,4.754759,10,-1.667170,-2.748589,1.892526,0.078780,0.02381
24996,41.531201,68.534175,2.921576,0.942888,64.620028,1.025187,42.577263,1.064813,3.110931,5.812807,10,-2.701876,-2.748589,1.892526,0.078780,0.02381
24997,41.531201,68.534175,2.921576,1.022187,70.054744,1.070474,44.458055,0.958255,2.799614,5.019518,10,-2.219905,-2.748589,1.892526,0.078780,0.02381
24998,41.531201,68.534175,2.921576,0.962832,65.986915,1.005164,41.745649,1.024074,2.991909,5.869096,10,-2.877187,-2.748589,1.892526,0.078780,0.02381


In [17]:
valores_unicos_fck = df['lambda 1'].unique()
print(valores_unicos_fck, len(valores_unicos_fck))

[  0.19328356 -15.76166736  -1.93365673  -0.66852264  -0.78180069
   0.77236132  -2.44010977   2.57688413  -3.22028596  -5.52847804
  -2.58793226  -9.12304442  -5.67881596  -1.67647677   0.03687997
  -1.31375663  -4.74163583  -6.97451031  -1.70105977  -4.22369038
 -11.48676067  -4.99877567   1.75800521  -7.97739866   0.27335869
  -9.79632329  -4.51505068  -5.00102733 -12.63191647  -1.48526847
  -0.45157545  -2.45080545  -2.51850248  -1.47722312  -3.03814116
 -15.37998123  -7.61500073   0.64485698  -1.31924793 -12.61729463
  -4.07966515  -6.8299915   -8.50026044  -2.80843034  -2.32624573
  -4.58339157  -9.31330395  -4.57641681  -1.10308191  -2.74858862] 50
